# Axis 1 — Unit conditionals (`10_unit_conditionals`)

**What this is.** The `gibbs_samplers` *output check*: every exact Gibbs
draw is compared to its analytic full conditional, re-derived **from the
model** in `_harness` (never from the code). This is the inspectable form
of `tests/test_unit_conditionals.py`.

**How to read it.** Each check freezes all conditioning variables, draws
`N_DRAWS` times, and prints the analytic mean/variance next to the sampled
mean/variance. A wrong mean but right variance points at the *linear
statistic* (a sign or missing `a`); a wrong variance points at the
*precision* (a wrong count or dropped `a²`). Read the two rows and see them
agree to within a few Monte-Carlo standard errors.

**Why frozen inputs are legitimate i.i.d. sampling** (not a chain): nothing
feeds back — every one of the `N_DRAWS` calls conditions on the *same* fixed
values, so all draws come from one fixed distribution, the analytic one.


## Setup


In [ ]:
import sys, numpy as np
sys.path.insert(0, '..'); sys.path.insert(0, '../..')
np.set_printoptions(precision=4, suppress=True)

def verdict(ok, label):
    print(f"  [{'PASS' if ok else 'FAIL'}]  {label}"); assert ok, label

def show_moment(draws, mu, var, label, n_se=4.0):
    draws = np.asarray(draws)
    emp_m = draws.mean(0); emp_v = draws.var(0, ddof=1)
    se = np.sqrt(draws.var(0, ddof=1) / draws.shape[0])
    zmax = float(np.max(np.abs(emp_m - mu) / np.where(se==0,1e-12,se)))
    print(f'  {label}')
    print(f'    analytic mean : {np.round(np.atleast_1d(mu),4)}')
    print(f'    sampled  mean : {np.round(np.atleast_1d(emp_m),4)}   (max {zmax:.2f} SE off)')
    print(f'    analytic var  : {np.round(np.atleast_1d(var),4)}')
    print(f'    sampled  var  : {np.round(np.atleast_1d(emp_v),4)}')
    verdict(zmax < n_se, label + f' mean within {n_se:.0f} SE')

N_DRAWS = 20000   # bump to 60000 for a strict read; lower for speed


In [ ]:
import irtree.gibbs_samplers as gs
import tests._harness as h


## 1. `gibbs_b` — the item-difficulty conditional

Precision `1/τ² + Σ_p a_i²`, statistic `Σ_p (a_i·θ_p − Z_pi)`. Under Rasch
(`a=1`) this is `1/τ² + n` and `Σ(θ − Z)`.


### 1a. 1PL


In [ ]:
rng = np.random.default_rng(1); P, I = 120, 6
theta = rng.normal(0,1,P); b = rng.normal(0,1,I)
T = rng.random((P,I)) > 0.25; ri, ci = np.where(T)
Z = rng.normal(h.simulate_probit_cell_means(theta, b), 1.0)
tau2 = 4.0
draws = h.draw_many(lambda r: gs.gibbs_b(Z, theta, tau2, T, ri, ci, r), N_DRAWS)
mu = np.empty(I); var = np.empty(I)
for i in range(I): mu[i], var[i] = h.analytic_b_scalar(Z[:,i], theta, T[:,i], tau2)
show_moment(draws, mu, var, 'gibbs_b (1PL)')


### 1b. 2PL — the `a`-weighting

Now the precision carries `a²` and the statistic carries `a`. If the code
dropped either, this is where it shows.


In [ ]:
rng = np.random.default_rng(2); a = rng.uniform(0.6,1.7,I)
Z2 = rng.normal(h.simulate_probit_cell_means(theta, b, a=a), 1.0)
draws = h.draw_many(lambda r: gs.gibbs_b(Z2, theta, tau2, T, ri, ci, r, a=a), N_DRAWS)
mu = np.empty(I); var = np.empty(I)
for i in range(I): mu[i], var[i] = h.analytic_b_scalar(Z2[:,i], theta, T[:,i], tau2, a_i=a[i])
show_moment(draws, mu, var, 'gibbs_b (2PL, a-weighted)')

# a=None must be bitwise identical to the 1PL call
r1 = gs.gibbs_b(Z, theta, tau2, T, ri, ci, np.random.default_rng(9))
r2 = gs.gibbs_b(Z, theta, tau2, T, ri, ci, np.random.default_rng(9), a=None)
verdict(np.array_equal(r1, r2), 'a=None bitwise identical to 1PL')


### 1c. The sign check

A strong (hard) item has a large positive `b`. If the statistic were
`Σ(Z − θ)` instead of `Σ(θ − Z)`, the posterior mean would land on the
*wrong side of zero*.


In [ ]:
rng = np.random.default_rng(7); Pp = 400
th = rng.normal(0,1,Pp); b_true = np.array([1.5])   # a hard item
Tc = np.ones((Pp,1), bool); ric, cic = np.where(Tc)
Zc = rng.normal(th[:,None] - b_true[None,:], 1.0)
draws = h.draw_many(lambda r: gs.gibbs_b(Zc, th, 4.0, Tc, ric, cic, r), 8000)
print(f'  posterior mean of b for a hard item: {draws.mean():.3f}  (expect ~ +1.5)')
verdict(draws.mean() > 0.5, 'b sign correct (statistic is theta - Z)')


## 2. θ via the cross-node path

There is no per-node `gibbs_theta`. `compute_suff_stats` →
`gibbs_theta_correlated` with a diagonal Σ is the independent case, and must
match the scalar posterior per person.


### 2a. 1PL θ


In [ ]:
rng = np.random.default_rng(4); sigma2 = 1.3
Sigma = gs.build_sigma(np.array([sigma2]), np.eye(1))
def draw_theta(r):
    S, n = gs.compute_suff_stats([Z],[b],[T],[ri],[ci],[False])
    S, n = gs.reduce_suff_stats_to_dims(S, n, [[0]])
    return gs.gibbs_theta_correlated(S, n, Sigma, r)[:,0]
draws = h.draw_many(draw_theta, N_DRAWS)
mu = np.empty(P); var = np.empty(P)
for p in range(P): mu[p], var[p] = h.analytic_theta_scalar(Z[p], b, T[p], sigma2)
emp = draws.mean(0); se = np.sqrt(draws.var(0,ddof=1)/N_DRAWS)
zmax = float(np.max(np.abs(emp-mu)/np.where(se==0,1e-12,se)))
print(f'  theta 1PL: max {zmax:.2f} SE off across {P} persons')
print(f'    example person 0 : analytic {mu[0]:.4f}  sampled {emp[0]:.4f}')
verdict(zmax < 4.5, 'theta 1PL matches (all persons)')


### 2b. GRM θ — the statistic must NOT include `+b`

For a polytomous node the mean of Z is θ (no `−b`), so the sufficient
statistic is `Σ Z`, not `Σ(Z + b)`. `is_polytomous=True` selects this.


In [ ]:
rng = np.random.default_rng(6)
theta_g = rng.normal(0,1,P); b_g = rng.normal(0,1,I)   # b present but must be ignored
Tg = rng.random((P,I)) > 0.25; rig, cig = np.where(Tg)
Zg = rng.normal(np.broadcast_to(theta_g[:,None],(P,I)).copy(), 1.0)
def draw_theta_grm(r):
    S, n = gs.compute_suff_stats([Zg],[b_g],[Tg],[rig],[cig],[True])
    S, n = gs.reduce_suff_stats_to_dims(S, n, [[0]])
    return gs.gibbs_theta_correlated(S, n, Sigma, r)[:,0]
draws = h.draw_many(draw_theta_grm, N_DRAWS)
mu = np.empty(P); var = np.empty(P)
for p in range(P):
    idx = np.where(Tg[p])[0]; v = 1.0/(1.0/sigma2 + len(idx))
    mu[p] = v*np.sum(Zg[p,idx]); var[p] = v
emp = draws.mean(0); se = np.sqrt(draws.var(0,ddof=1)/N_DRAWS)
zmax = float(np.max(np.abs(emp-mu)/np.where(se==0,1e-12,se)))
print(f'  theta GRM (no +b): max {zmax:.2f} SE off')
verdict(zmax < 4.5, 'GRM theta statistic omits b')


### 2c. Correlated θ — the full MVN conditional

With a non-diagonal Σ, each person's draw is a genuine MVN. Check mean and
covariance for a few persons.


In [ ]:
rng = np.random.default_rng(11); Pc, D = 30, 3
C = np.full((D,D),0.4); np.fill_diagonal(C,1.0)
sd = np.sqrt(np.linspace(0.8,1.3,D)); Sig = np.diag(sd)@C@np.diag(sd)
n = rng.integers(2,12,(Pc,D)).astype(float); S = rng.normal(0, np.sqrt(n+1))
draws = h.draw_many(lambda r: gs.gibbs_theta_correlated(S, n, Sig, r), N_DRAWS)
p = 0; mu, V = h.analytic_theta_mvn(S[p], n[p], Sig)
print(f'  person {p}:')
print('    analytic mean', np.round(mu,4), ' sampled', np.round(draws[:,p,:].mean(0),4))
print('    analytic cov\n', np.round(V,4)); print('    sampled  cov\n', np.round(np.cov(draws[:,p,:].T),4))
h.assert_mean(draws[:,p,:], mu, label='theta_corr mean p0')
h.assert_cov(draws[:,p,:], V, label='theta_corr cov p0')
print('  [PASS]  correlated theta MVN (person 0)')


## 3. Covariate coefficient — `gibbs_coef_pooled` vs brute-force regression

A coefficient is a Bayesian linear regression. Build the design (`X_p` per
relevant cell) and residual `y = Z − (θ − b)` by hand, get the exact
posterior from `V = (I/τ² + DᵀD)⁻¹`, and check the sampler reproduces it.


In [ ]:
rng = np.random.default_rng(41); Pp2, Ii2, K = 200, 12, 2
theta_c = rng.normal(0,1,Pp2); b_c = rng.normal(0,1,Ii2)
X = rng.normal(size=(Pp2,K)); beta = rng.normal(size=K)
Tc2 = rng.random((Pp2,Ii2)) > 0.25; ric2, cic2 = np.where(Tc2)
means = theta_c[ric2] - b_c[cic2] + (X @ beta)[ric2]
Zc2 = np.zeros((Pp2,Ii2)); Zc2[ric2,cic2] = rng.normal(means, 1.0)
tau2b = 3.0; Xobs = np.ones(Pp2, bool)
# run_chain draws a coefficient from each node's latent residual with
# gibbs_coef_pooled(kind, nodes, residuals, masks, discriminations, design, tau2, rng);
# kind 'x' = person covariates (beta), 'q' = item covariates (kappa).
E = np.zeros((Pp2,Ii2)); E[ric2,cic2] = Zc2[ric2,cic2] - (theta_c[ric2] - b_c[cic2])   # latent residual
draws = h.draw_many(lambda r: gs.gibbs_coef_pooled('x', [0], [E], [Tc2], [None], X, tau2b, r), N_DRAWS)
y = Zc2[ric2,cic2] - (theta_c[ric2] - b_c[cic2]); design = X[ric2]
mu, V = h.analytic_normal_normal(design, y, tau2b)
print('  analytic beta mean :', np.round(mu,4))
print('  sampled  beta mean :', np.round(draws.mean(0),4))
print('  analytic beta cov  :\n', np.round(V,5))
print('  sampled  beta cov  :\n', np.round(np.cov(draws.T),5))
h.assert_mean(draws, mu, label='beta'); print('  [PASS]  gibbs_coef_pooled (beta) matches brute-force design')


## 4. Discrimination `a` — the offset must be in the regressor

`gibbs_a` regresses `Z` on the ability predictor. With covariates that
predictor is `θ − b + offset`. Drop the offset and `a` absorbs the covariate
signal. Here we check the *with-offset* draw against the truncated-Normal
posterior it should have.


In [ ]:
rng = np.random.default_rng(72); Pa, Ia = 400, 6
th_a = rng.normal(0,1,Pa); b_a = rng.normal(0,1,Ia)
a_true = rng.uniform(0.7,1.6,Ia); offset = rng.normal(0,0.8,(Pa,Ia))
Ta = np.ones((Pa,Ia), bool); ria, cia = np.where(Ta)
Za = rng.normal(a_true[None,:]*(th_a[:,None]-b_a[None,:]+offset), 1.0)
mean_a, var_a = 1.0, 0.5
draws = h.draw_many(lambda r: gs.gibbs_a(Za, th_a, b_a, ria, cia, mean_a, var_a, 0.0,
        r, n_items=Ia, offset=offset), N_DRAWS)
print('  item : analytic trunc-normal mean  vs  sampled mean')
for i in range(Ia):
    d_pi = th_a - b_a[i] + offset[:,i]              # offset IN the regressor
    prec = 1.0/var_a + np.sum(d_pi**2); v = 1.0/prec
    mu = v*(mean_a/var_a + np.sum(Za[:,i]*d_pi))
    tm,_ = h.truncnorm_moments(mu, np.sqrt(v), lower=0.0)
    print(f'    {i}: {tm:.4f}   {draws[:,i].mean():.4f}')
    h.assert_mean(draws[:,i:i+1], [tm], n_se=5.0, label=f'a offset i={i}')
print('  [PASS]  gibbs_a with offset matches truncated-Normal posterior')


## Summary

Every conditional's sampled moments match the analytically-derived moments
within Monte-Carlo error — so the sufficient statistics (precisions, linear
statistics, the `a`-weighting, the signs, the offset) are written correctly.
These are the pieces; that they *compose* into a correct chain is Axis 2.
